In [ ]:
# Run once to convert raw CSV to Parquet

import pandas as pd

cols = [
    "EPOCH_SECONDS", "CURRENCY_CODE", "USD_AMOUNT", "ORDERING_BIC8",
    "BENEFICIARY_BIC8", "BIC8_PATH", "TOTAL_DURATION", "CONTINUITY_QFLAG",
    "CURRENCY_CONVERSION_COUNT", "PARTICIPANT_COUNT", "TRANSACTION_TYPE_CODE",
    "ORDERING_TYPE", "BENEFICIARY_TYPE", "ORDERING_ACCOUNT", "BENEFICIARY_ACCOUNT",
    "ANOMALY_TYPE"  # Included so you can verify your rules against the labels!
]

df = pd.read_csv(
    "data/swift_umdac_datathon_2026_fictitious_payment_data.csv",
    usecols=cols
)

# Optional: Save your own local Parquet copy right away so future runs load in 2 seconds!
df.to_parquet("data/clean_data.parquet", index=False)

C:\Users\user\AppData\Local\Temp\ipykernel_10880\73269652.py:11: DtypeWarning: Columns (20: ANOMALY_TYPE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [3]:
df = pd.read_parquet("data/clean_data.parquet")
print(df.shape)
print(df["ANOMALY_TYPE"].value_counts(dropna=False))

(10000000, 16)
ANOMALY_TYPE
NaN                         9992994
ROMANCE_SCAM                   2476
FAT_FINGER_MISTAKE             1940
CEO_SCAM                        945
DUPLICATE_PAYMENT               679
ACCOUNT_REPLACEMENT_SCAM        649
CUSTOMER_RECALL                 317
Name: count, dtype: int64


In [ ]:
# Separate anomalies and normal baseline
anomalies = df[df["ANOMALY_TYPE"].notna()].copy()
normal_sample = df[df["ANOMALY_TYPE"].isna()].sample(n=50000, random_state=42).copy()

combined = pd.concat([anomalies, normal_sample], ignore_index=True)

# Keep ONLY assigned columns: party types, currencies, and routing behaviour
your_cols = [
    'ANOMALY_TYPE',
    # Party types
    'ORDERING_TYPE', 'BENEFICIARY_TYPE',
    # Currencies & FX
    'CURRENCY_CODE', 'USD_AMOUNT', 'CURRENCY_CONVERSION_COUNT',
    # Routing behaviour
    'ORDERING_BIC8', 'BENEFICIARY_BIC8', 'BIC8_PATH',
    'TOTAL_DURATION', 'CONTINUITY_QFLAG', 'PARTICIPANT_COUNT',
    'TRANSACTION_TYPE_CODE'
]

slice_df = combined[your_cols]
slice_df.to_csv("claude_routing_currency_party.csv", index=False)
print(f"Done! Saved {len(slice_df)} rows to claude_routing_currency_party.csv")

Done! Saved 57006 rows to claude_routing_currency_party.csv


In [6]:
import os
import pandas as pd
import numpy as np

# Create outputs folder if it doesn't exist
os.makedirs("outputs", exist_ok=True)

df = pd.read_parquet("data/clean_data.parquet")
df["CLASS"] = df["ANOMALY_TYPE"].fillna("NORMAL_BASELINE")

# 1. FULL 10M STRUCTURAL INVARIANTS CHECK
print("--- 1. FULL 10M STRUCTURAL INVARIANTS ---")
has_gap = df["BIC8_PATH"].str.contains(r"\.\.\.", na=False)
flag_is_n = df["CONTINUITY_QFLAG"] == "N"
inv_continuity = (has_gap == flag_is_n).mean() * 100

is_direct = df["TRANSACTION_TYPE_CODE"] == "Direct"
part_is_2 = df["PARTICIPANT_COUNT"] == 2
inv_direct = (is_direct == part_is_2).mean() * 100

# Extract first and last bank from BIC8_PATH to check endpoint mismatches
first_bic = df["BIC8_PATH"].str.split(r"[,\-\>\s]+").str[0]
last_bic = df["BIC8_PATH"].str.split(r"[,\-\>\s]+").str[-1]
df["SENDER_MISMATCH"] = first_bic != df["ORDERING_BIC8"]
df["BENEFICIARY_MISMATCH"] = last_bic != df["BENEFICIARY_BIC8"]

print(f"CONTINUITY_QFLAG == 'N' <-> '...' in BIC8_PATH: {inv_continuity:.4f}% match")
print(f"TRANSACTION_TYPE_CODE == 'Direct' <-> PARTICIPANT_COUNT == 2: {inv_direct:.4f}% match")

invariants_df = pd.DataFrame({
    "Invariant_Rule": [
        "CONTINUITY_QFLAG == 'N' iff '...' gap in BIC8_PATH",
        "TRANSACTION_TYPE_CODE == 'Direct' iff PARTICIPANT_COUNT == 2"
    ],
    "Total_Rows_Tested": [len(df), len(df)],
    "Exact_Match_Pct": [round(inv_continuity, 4), round(inv_direct, 4)]
})
invariants_df.to_csv("outputs/structural_invariants_10m.csv", index=False)


# 2. PARTY TYPE MATRIX (B->C, B->B, C->C, C->B)
df["PARTY_PAIR"] = df["ORDERING_TYPE"] + "->" + df["BENEFICIARY_TYPE"]
party_ct = pd.crosstab(df["CLASS"], df["PARTY_PAIR"], normalize="index") * 100
party_counts = df["CLASS"].value_counts().rename("TOTAL_ROWS")
party_summary = party_ct.round(2).join(party_counts).reset_index()
party_summary.to_csv("outputs/party_type_matrix.csv", index=False)
print("\n--- 2. PARTY TYPE MATRIX (%) ---")
print(party_summary.to_string(index=False))


# 3. ROUTING & SCAM SMOKESCREEN SUMMARY
routing_summary = df.groupby("CLASS").agg(
    TOTAL_ROWS=("CLASS", "count"),
    DIRECT_PCT=("TRANSACTION_TYPE_CODE", lambda s: round((s == "Direct").mean() * 100, 2)),
    INDIRECT_PCT=("TRANSACTION_TYPE_CODE", lambda s: round((s == "Indirect").mean() * 100, 2)),
    COVERED_PCT=("TRANSACTION_TYPE_CODE", lambda s: round((s == "Covered").mean() * 100, 2)),
    CONTINUITY_N_PCT=("CONTINUITY_QFLAG", lambda s: round((s == "N").mean() * 100, 2)),
    SENDER_MISMATCH_PCT=("SENDER_MISMATCH", lambda s: round(s.mean() * 100, 2)),
    BENEFICIARY_MISMATCH_PCT=("BENEFICIARY_MISMATCH", lambda s: round(s.mean() * 100, 2)),
    AVG_PARTICIPANTS=("PARTICIPANT_COUNT", lambda s: round(s.mean(), 2)),
    MEDIAN_DURATION_SEC=("TOTAL_DURATION", "median")
).reset_index()

# Also compute Covered share specifically among multi-hop (>2 banks) transactions
multihop = df[df["PARTICIPANT_COUNT"] > 2]
cov_multihop = multihop.groupby("CLASS")["TRANSACTION_TYPE_CODE"].apply(
    lambda s: round((s == "Covered").mean() * 100, 2)
).rename("COVERED_AMONG_MULTIHOP_PCT").reset_index()

routing_summary = pd.merge(routing_summary, cov_multihop, on="CLASS", how="left")
routing_summary.to_csv("outputs/routing_by_anomaly.csv", index=False)
print("\n--- 3. ROUTING & MISMATCH SUMMARY ---")
print(routing_summary.to_string(index=False))


# 4. BIMODAL DURATION & HOP BREAKDOWN
df["DURATION_MODE"] = np.where(df["TOTAL_DURATION"] <= 300, "Fast_Mode (<=5m)", "Slow_Mode (>5m)")
dur_summary = df.groupby(["PARTICIPANT_COUNT", "DURATION_MODE"]).agg(
    TXN_COUNT=("TOTAL_DURATION", "count"),
    MEDIAN_SEC=("TOTAL_DURATION", "median"),
    P90_SEC=("TOTAL_DURATION", lambda s: round(np.percentile(s, 90), 1))
).reset_index()
dur_summary.to_csv("outputs/duration_bimodal_stats.csv", index=False)
print("\n--- 4. BIMODAL DURATION BY HOP COUNT ---")
print(dur_summary.head(12).to_string(index=False))

print("\nAll 4 summary CSVs saved cleanly to 'outputs/'!")

--- 1. FULL 10M STRUCTURAL INVARIANTS ---
CONTINUITY_QFLAG == 'N' <-> '...' in BIC8_PATH: 100.0000% match
TRANSACTION_TYPE_CODE == 'Direct' <-> PARTICIPANT_COUNT == 2: 100.0000% match

--- 2. PARTY TYPE MATRIX (%) ---
                   CLASS  B->B  B->C  C->B  C->C  TOTAL_ROWS
ACCOUNT_REPLACEMENT_SCAM 21.26 57.01  5.24 16.49         649
                CEO_SCAM 27.72 72.28  0.00  0.00         945
         CUSTOMER_RECALL 19.87 57.73  6.94 15.46         317
       DUPLICATE_PAYMENT 22.39 57.88  3.09 16.64         679
      FAT_FINGER_MISTAKE 22.94 56.60  5.21 15.26        1940
         NORMAL_BASELINE 22.71 56.05  6.10 15.13     9992994
            ROMANCE_SCAM  0.00  0.00 26.86 73.14        2476

--- 3. ROUTING & MISMATCH SUMMARY ---
                   CLASS  TOTAL_ROWS  DIRECT_PCT  INDIRECT_PCT  COVERED_PCT  CONTINUITY_N_PCT  SENDER_MISMATCH_PCT  BENEFICIARY_MISMATCH_PCT  AVG_PARTICIPANTS  MEDIAN_DURATION_SEC  COVERED_AMONG_MULTIHOP_PCT
ACCOUNT_REPLACEMENT_SCAM         649       68.2